# NSE All Stocks LSTM Predictor - Colab GPU

## Features:
- All NSE Stocks - Comprehensive stock list
- GPU Optimized - Optimized for Google Colab
- REST API - Get predictions via HTTP
- Model Save/Load - Reuse trained model

## Usage:
```python
predict_stock('RELIANCE')
```

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted!")

In [ ]:
# Install dependencies
!pip install yfinance tensorflow scikit-learn pandas numpy matplotlib flask flask-cors -q
print("Done!")

In [ ]:
# Imports
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import pickle
import json
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

import tensorflow as tf
print(f"TensorFlow: {tf.__version__}")
print(f"GPU: {tf.config.list_physical_devices('GPU')}")

from tensorflow.keras.models import Sequential, load_model as tf_load_model
from tensorflow.keras.layers import LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

## NSE Stocks List

In [ ]:
# NSE Stocks - Comprehensive List
nse_stocks_raw = [
    'RELIANCE', 'TCS', 'HDFCBANK', 'INFY', 'ICICIBANK',
    'HINDUNILVR', 'SBIN', 'BHARTIARTL', 'KOTAKBANK', 'ITC',
    'LT', 'AXISBANK', 'ASIANPAINT', 'MARUTI', 'SUNPHARMA',
    'TITAN', 'BAJFINANCE', 'ULTRACEMCO', 'NESTLEIND', 'WIPRO',
    'HCLTECH', 'TECHM', 'POWERGRID', 'NTPC', 'M&M',
    'ADANIPORTS', 'ONGC', 'BPCL', 'COALINDIA', 'DRREDDY',
    'CIPLA', 'DIVISLAB', 'GRASIM', 'HEROMOTOCO', 'INDUSINDBK',
    'JSWSTEEL', 'TATASTEEL', 'UPL', 'ADANIENT', 'TATAMOTORS',
    'AUBANK', 'BAJAJ-AUTO', 'BAJAJFINSV', 'BANDHANBNK', 'BANKBARODA',
    'BERGEPAINT', 'BIOCON', 'CANBK', 'CHOLAFIN', 'DABUR',
    'DLF', 'DMART', 'EICHERMOT', 'GAIL', 'GODREJCP',
    'HAVELLS', 'HDFCLIFE', 'ICICIGI', 'ICICIPRULI', 'IDFCFIRSTB',
    'INDHOTELS', 'JUBLFOOD', 'LICHSGFIN', 'MFSL', 'MGL',
    'MOTHERSON', 'MPHASIS', 'MRF', 'MUTHOOTFIN', 'NAUKRI',
    'NMDC', 'PAGEIND', 'PERSISTENT', 'PETRONET', 'PFC',
    'PIDILITIND', 'PNB', 'POLYCAB', 'RAIN', 'RBLBANK',
    'SBICARD', 'SBILIFE', 'SHRIRAMFIN', 'SIEMENS', 'TATACONSUM',
    'TATAPOWER', 'TORNTPHARM', 'TVSMOTOR', 'UBL', 'UJJIVAN',
    'UNIONBANK', 'VBL', 'VEDL', 'VOLTAS', 'ZOMATO',
    'ACC', 'ADANIGREEN', 'ADANIPOWER', 'ALEMBICLTD', 'ALKEM',
    'AMARAJABAT', 'ASHOKLEY', 'ASTRAL', 'ATUL', 'AUROPHARMA',
    'AVANTIFEED', 'BATAINDIA', 'BEL', 'BEML', 'BHEL',
    'BRITANNIA', 'CANFINHOME', 'CASTROLIND', 'CEATLTD', 'CENTRALBK',
    'CENTURYPLY', 'CHOLAHLDNG', 'COLPAL', 'CROMPTON', 'CUB',
    'DALBHARAT', 'DEEPAKFERT', 'DEEPAKNTR', 'DELHIVERY', 'DIXON',
    'DREAMFOLKS', 'ENDURANCE', 'ESCORTS', 'EXIDEIND', 'FEDERALBNK',
    'FINEORG', 'FORCEMOTORS', 'FORTIS', 'FSL', 'GLENMARK',
    'GMMPFAUDLR', 'GNFC', 'GODREJIND', 'GODREJPROP', 'GRINDWELL',
    'HAL', 'HINDPETRO', 'HONAUT', 'HSCL', 'IBULHSGFIN',
    'ICICI', 'IDBI', 'IGL', 'INDROCEM', 'INTELLECT',
    'IOB', 'IRB', 'IRCON', 'IRCTC', 'IRFC',
    'JINDALSTEL', 'JKCEMENT', 'JKLAKSHMI', 'KAJARIA', 'KALYAN',
    'KEC', 'KEI', 'L&TFH', 'LALPATHLAB', 'LAURUSLABS',
    'LUMAXIND', 'LUPIN', 'M&MFIN', 'MACPOWER', 'MAHABANK',
    'MAHINDCIE', 'MAHLIFE', 'MANAPPURAM', 'MAXHEALTH', 'METROBRAND',
    'MFSL', 'MIDHANI', 'MOTILALOFS', 'MRF', 'MUTHOOTFIN',
    'NATIONALUM', 'NAVINFLUOR', 'NBCC', 'NESTLE', 'NLCINDIA',
    'NMDC', 'NRAIL', 'NTPC', 'OIL', 'OLECTRA',
    'ORIENTELEC', 'PERSISTENT', 'PFC', 'PHILIPCARB', 'PIIND',
    'PNBHOUSING', 'PNCINFRA', 'PRESTIGE', 'QUESS', 'RAJESHEXP',
    'RAMCOCEM', 'RATNAMANI', 'RECLTD', 'SAIL', 'SANDHAR',
    'SCHAEFFLER', 'SFL', 'SHREECEM', 'SHRIRAMFIN', 'SIEMENS',
    'SOLARINDS', 'SONATACOM', 'SRF', 'STAR', 'STARCEMENT',
    'SUDARSCHEM', 'SUNDRMCEM', 'SUPREMEIND', 'TATACHEM', 'TATACOMM',
    'TATAELXSI', 'TATAINVEST', 'TATAMTRD', 'TATASTLBSL', 'TCS',
    'TECHM', 'THERMAX', 'TIMKEN', 'TITAN', 'TORNTPHARM',
    'TRENT', 'TRITURB', 'TV18BRDCST', 'TVSSRICITY', 'UBL',
    'UJJIVAN', 'ULTRAMARINE', 'UNIONBANK', 'UNO Mena', 'UPL',
    'VAKRANGEE', 'VARROC', 'VEDL', 'VINATIPHARMA', 'VOLTAS',
    'WHIRLPOOL', 'WIPRO', 'ZENTEC'
]

# Remove duplicates
nse_stocks_raw = list(dict.fromkeys(nse_stocks_raw))
nse_stocks = [f"{s}.NS" for s in nse_stocks_raw]

print(f"Total stocks in list: {len(nse_stocks)}")
print(f"Sample: {nse_stocks[:5]}")

## Configuration

In [ ]:
# CONFIGURATION - Change STOCK_LIMIT to control training size

STOCK_LIMIT = 100

valid_stocks = nse_stocks[:STOCK_LIMIT]
print(f"Stocks to train: {len(valid_stocks)}")

# Date range (5 years)
end_date = datetime.now()
start_date = end_date - timedelta(days=5*365)

# Model parameters
SEQUENCE_LENGTH = 60
TEST_SPLIT = 0.2
EPOCHS = 50
BATCH_SIZE = 64
LEARNING_RATE = 0.001

# Save paths
SAVE_PATH = '/content/drive/MyDrive/Colab Notebooks/nse_lstm_model'
os.makedirs(SAVE_PATH, exist_ok=True)

MODEL_PATH = f'{SAVE_PATH}/model.h5'
SCALER_X = f'{SAVE_PATH}/scaler_x.pkl'
SCALER_Y = f'{SAVE_PATH}/scaler_y.pkl'
DATA_PATH = f'{SAVE_PATH}/data.pkl'
META_PATH = f'{SAVE_PATH}/meta.json'

print(f"Date range: {start_date.date()} to {end_date.date()}")
print(f"Save path: {SAVE_PATH}")

## Download Data

In [ ]:
print(f"Downloading data for {len(valid_stocks)} stocks...")

data = yf.download(
    valid_stocks,
    start=start_date,
    end=end_date,
    group_by='ticker',
    auto_adjust=True,
    progress=True,
    threads=True
)

print(f"Downloaded shape: {data.shape}")

In [ ]:
# Extract closing prices
close_data = {}
for ticker in valid_stocks:
    try:
        if len(valid_stocks) == 1:
            prices = data['Close']
        else:
            prices = data[ticker]['Close']
        if not prices.isna().all():
            close_data[ticker] = prices
    except:
        pass

data_pivot = pd.DataFrame(close_data).dropna()

print(f"Clean data: {data_pivot.shape}")
print(f"Stocks: {data_pivot.shape[1]}, Days: {data_pivot.shape[0]}")

## Prepare Data

In [ ]:
def create_sequences(data, seq_len):
    X, y = [], []
    for i in range(seq_len, len(data)):
        X.append(data[i-seq_len:i])
        y.append(data[i])
    return np.array(X), np.array(y)

scaler_x = MinMaxScaler(feature_range=(0, 1))
scaler_y = MinMaxScaler(feature_range=(0, 1))

prices = data_pivot.values
prices_scaled = scaler_x.fit_transform(prices)

X, y = create_sequences(prices_scaled, SEQUENCE_LENGTH)
scaler_y.fit(prices)
y_scaled = scaler_y.transform(y)

split_idx = int(len(X) * (1 - TEST_SPLIT))
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y_scaled[:split_idx], y_scaled[split_idx:]

n_features = X_train.shape[2]
n_outputs = y_train.shape[1]

print(f"Train: {X_train.shape[0]}, Test: {X_test.shape[0]}")
print(f"Input: {X_train.shape[1:]}, Output: {y_train.shape[1:]}")

## Build Model

In [ ]:
model = Sequential([
    Bidirectional(LSTM(128, return_sequences=True), input_shape=(SEQUENCE_LENGTH, n_features)),
    Dropout(0.2),
    Bidirectional(LSTM(64, return_sequences=True)),
    Dropout(0.2),
    LSTM(32, return_sequences=False),
    Dropout(0.2),
    Dense(64, activation='relu'),
    Dense(n_outputs)
])

model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.summary()

## Train

In [ ]:
callbacks = [
    EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=0.00001, verbose=1),
    ModelCheckpoint(MODEL_PATH, monitor='val_loss', save_best_only=True, verbose=1)
]

print("Training...")
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1
)
print("Done!")

## Evaluate

In [ ]:
y_pred = model.predict(X_test)
y_test_actual = scaler_y.inverse_transform(y_test)
y_pred_actual = scaler_y.inverse_transform(y_pred)

mae = mean_absolute_error(y_test_actual, y_pred_actual)
rmse = np.sqrt(mean_squared_error(y_test_actual, y_pred_actual))
mape = np.mean(np.abs((y_test_actual - y_pred_actual) / (y_test_actual + 1e-8))) * 100

print(f"MAE: ₹{mae:.2f}")
print(f"RMSE: ₹{rmse:.2f}")
print(f"MAPE: {mape:.2f}%")

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Training History')
plt.legend()
plt.savefig(f'{SAVE_PATH}/history.png')
plt.show()

## Save Model

In [ ]:
with open(SCALER_X, 'wb') as f:
    pickle.dump(scaler_x, f)
with open(SCALER_Y, 'wb') as f:
    pickle.dump(scaler_y, f)
with open(DATA_PATH, 'wb') as f:
    pickle.dump(data_pivot, f)

meta = {
    'stocks': list(data_pivot.columns),
    'seq_len': SEQUENCE_LENGTH,
    'trained': datetime.now().isoformat()
}
with open(META_PATH, 'w') as f:
    json.dump(meta, f)

print(f"Saved to {SAVE_PATH}")

## Prediction Functions

In [ ]:
prices_scaled = scaler_x.transform(data_pivot.values)
stock_cols = list(data_pivot.columns)

def predict_stock(symbol):
    symbol = symbol if symbol.endswith('.NS') else f"{symbol}.NS"
    if symbol not in stock_cols:
        return f"Stock {symbol} not found"
    idx = stock_cols.index(symbol)
    seq = prices_scaled[-SEQUENCE_LENGTH:].reshape(1, SEQUENCE_LENGTH, n_features)
    pred = model.predict(seq, verbose=0)
    pred_price = scaler_y.inverse_transform(pred)[0, idx]
    last_price = scaler_y.inverse_transform([prices_scaled[-1]])[0, idx]
    change = ((pred_price - last_price) / last_price) * 100
    return {
        'symbol': symbol,
        'last': round(last_price, 2),
        'predicted': round(pred_price, 2),
        'change_pct': round(change, 2)
    }

print("Ready to predict!")

In [ ]:
# Test predictions
for s in ['RELIANCE', 'TCS', 'INFY', 'HDFCBANK']:
    r = predict_stock(s)
    print(f"{r['symbol']}: ₹{r['predicted']} ({r['change_pct']:+.2f}%)")

## REST API

In [ ]:
from flask import Flask, jsonify, request
from flask_cors import CORS

app = Flask(__name__)
CORS(app)

@app.route('/predict/<path:symbol>')
def api_predict(symbol):
    return jsonify(predict_stock(symbol))

@app.route('/stocks')
def api_stocks():
    return jsonify({'stocks': stock_cols})

@app.route('/health')
def api_health():
    return jsonify({'status': 'ok'})

def run_api(port=5000):
    print(f"API running on http://localhost:{port}")
    app.run(host='0.0.0.0', port=port)

# run_api()

## Summary

```python
# Predict stock
predict_stock('RELIANCE')

# Run API
run_api(port=5000)
```

**Files saved:**
- model.h5
- scaler_x.pkl
- scaler_y.pkl
- data.pkl
- meta.json